# 06 · Validation choice and release audit

**Job:** pick the retrieval setup on the 100 validation images, then sign off the preprocessing release.

**Part A — validation (brief §4.4.4).** No human relevance labels exist, so we use a proxy:
for each validation image, take the top-5 KB images and compare object categories.

- **Jaccard@5** (main): mean Jaccard between the query's COCO categories and each retrieved image's categories.
- **Category recall@5**: share of the query's categories found in the top-5.
- Each setup is compared with the current one (hybrid, RRF k = 60, current BM25 cleaning) using a paired bootstrap 95% CI (2,000 resamples).

Setups tried: single sources, dense-only fusion, RRF k ∈ {10, 30, 60, 100}, and 3 BM25 tokenisers. Test images are not touched.

**Part B — release audit.** Every automated check on every artifact, then the release files.

| Output | Where |
|---|---|
| All setups, scores, CIs | `results/preprocessing_release/validation_selection.csv` |
| Decision | `results/preprocessing_release/validation_selection.json` |
| Audit, hashes, README | `results/preprocessing_release/` |

Run on Google Colab. All files are read from and written to `MyDrive/G9_Hybrid_RAG/`. Needs notebooks 01–05. **Use a T4 GPU** for Part A.

## Part A · Validation choice

### A1. Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
%pip -q install faiss-cpu rank-bm25 timm

Mounted at /content/drive


In [ ]:
import json, re, time, zipfile
from io import BytesIO
from pathlib import Path
from itertools import combinations
from collections import defaultdict
import numpy as np, pandas as pd, torch, faiss, transformers
from PIL import Image, ImageOps
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from transformers import CLIPModel, CLIPImageProcessor, AutoTokenizer

SEED = 42
rng = np.random.default_rng(SEED)
ROOT = Path('/content/drive/MyDrive/G9_Hybrid_RAG')
DATA = ROOT/'data'
EMB = DATA/'embeddings/clip_vit_base_patch32'
IDX = DATA/'indexes'
ARCH = DATA/'images_archives'
QA = ROOT/'results/qa'
RELEASE = ROOT/'results/preprocessing_release'
META = DATA/'metadata'
for d in (QA, RELEASE, META): d.mkdir(parents=True, exist_ok=True)

with open(DATA/'splits/split_ids.json') as f: splits = json.load(f)
SPLITS = ['knowledge_base', 'validation', 'test']
ids = {s: sorted(map(int, splits[s])) for s in SPLITS}
manifest = pd.read_csv(DATA/'splits/image_manifest.csv')
fname = {int(r.image_id): f"{r.split}/{r.file_name}" for r in manifest.itertuples()}
split_of = {int(r.image_id): r.split for r in manifest.itertuples()}

caps = pd.read_csv(EMB/'kb_text_metadata.csv')          # row i <-> text embedding i <-> text index position i
img_ids = np.load(EMB/'kb_image_ids.npy')               # row i <-> image embedding i <-> image index position i
img_emb = np.load(EMB/'kb_image_embeddings.npy')
txt_emb = np.load(EMB/'kb_text_embeddings.npy')
assert img_emb.shape == (10000, 512) and txt_emb.shape == (50000, 512) and len(caps) == 50000
assert (np.diff(img_ids) > 0).all(), 'image ids must be sorted ascending'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
CLIP_NAME = 'openai/clip-vit-base-patch32'
clip = CLIPModel.from_pretrained(CLIP_NAME).to(device).eval()
proc = CLIPImageProcessor.from_pretrained(CLIP_NAME)
tok = AutoTokenizer.from_pretrained(CLIP_NAME)

def feat(o):
    # transformers 4.x returns a tensor, 5.x returns an output object; handle both
    return o.pooler_output if hasattr(o, 'pooler_output') else o

@torch.inference_mode()
def embed_images(images, bs=32):
    out = []
    for i in range(0, len(images), bs):
        px = proc(images=images[i:i+bs], return_tensors='pt')['pixel_values'].to(device)
        f = feat(clip.get_image_features(pixel_values=px)).float()
        out.append(torch.nn.functional.normalize(f, dim=-1).cpu().numpy())
    return np.concatenate(out).astype('float32')

@torch.inference_mode()
def embed_texts(texts, bs=128):
    out = []
    for i in range(0, len(texts), bs):
        t = tok(texts[i:i+bs], padding=True, truncation=True, max_length=77, return_tensors='pt').to(device)
        f = feat(clip.get_text_features(**t)).float()
        out.append(torch.nn.functional.normalize(f, dim=-1).cpu().numpy())
    return np.concatenate(out).astype('float32')

def zip_index(split):
    d = {}
    for zp in sorted((ARCH/split).glob('part_*.zip')):
        with zipfile.ZipFile(zp) as z:
            for n in z.namelist():
                if not n.endswith('/'): d[n] = zp
    return d
ZIPS = {s: zip_index(s) for s in SPLITS}

def load_image(image_id):
    name = fname[image_id]
    with zipfile.ZipFile(ZIPS[name.split('/')[0]][name]) as z, Image.open(BytesIO(z.read(name))) as im:
        return ImageOps.exif_transpose(im).convert('RGB')

print('device:', device, '| transformers', transformers.__version__)
print({s: len(ids[s]) for s in SPLITS})

device: cuda | transformers 5.22.1
{'knowledge_base': 10000, 'validation': 100, 'test': 100}


In [ ]:
# Load the two FAISS indexes used by the search functions below.
img_index = faiss.read_index(str(IDX/'faiss_images.index'))
txt_index = faiss.read_index(str(IDX/'faiss_captions.index'))
assert img_index.ntotal == 10000 and txt_index.ntotal == 50000
print('indexes loaded')

### A2. Helpers

Scores, RRF, bootstrap, and the 3 BM25 tokenisers. Pure Python, no GPU.

In [ ]:
# ---------- pure helpers (no GPU / Drive needed) ----------
def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if (a | b) else 0.0

def score_list(ranked_ids, gt_cats, kb_cats, k=5):
    top = list(ranked_ids)[:k]
    if not gt_cats:                       # no GT objects -> undefined, excluded from means
        return np.nan, np.nan
    if not top:
        return 0.0, 0.0
    jac = float(np.mean([jaccard(gt_cats, kb_cats[i]) for i in top]))
    union = set().union(*[set(kb_cats[i]) for i in top])
    return jac, len(set(gt_cats) & union) / len(set(gt_cats))

def dedupe_to_images(positions, pos_to_image):
    seen, out = set(), []
    for p in positions:
        i = int(pos_to_image[int(p)])
        if i not in seen:
            seen.add(i); out.append(i)
    return out

def rrf(rankings, k=60):
    s = defaultdict(float)
    for ranked in rankings:
        for r, i in enumerate(ranked, 1):
            s[i] += 1.0 / (k + r)
    return sorted(s, key=lambda i: (-s[i], i))        # tie-break on id -> deterministic

def paired_bootstrap(diff, n=2000, seed=SEED):
    diff = np.asarray(diff, float); diff = diff[~np.isnan(diff)]
    r = np.random.default_rng(seed)
    means = np.array([r.choice(diff, len(diff), replace=True).mean() for _ in range(n)])
    return float(diff.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

_AP = r"['\u2019]"
TOKENIZERS = {
    'current_clean':    lambda t: re.findall(r"\w+", t.lower()),                       # = notebook 01 clean_caption + split
    'keep_apostrophes': lambda t: re.findall(r"[a-z0-9]+(?:" + _AP + r"[a-z]+)?", t.lower()),
    'no_stopwords':     lambda t: [w for w in re.findall(r"\w+", t.lower()) if w not in ENGLISH_STOP_WORDS],
}

def bm25_rank(bm25, tokenizer, query, pos_to_image, top=300):
    q = tokenizer(query)
    if not q: return []
    sc = bm25.get_scores(q)
    if sc.max() <= 0: return []
    order = np.argsort(sc)[::-1][:top]
    order = [p for p in order if sc[p] > 0]
    return dedupe_to_images(order, pos_to_image)

### A3. Validation inputs

Categories come from COCO labels of the validation images. The BM25 query is a BLIP caption, exactly as in the real pipeline.

In [ ]:
# ---------- validation data ----------
with open(DATA/'annotations/instances_val2017.json') as f: val_inst = json.load(f)
vcat = {c['id']: c['name'] for c in val_inst['categories']}
VAL = set(ids['validation'])
val_cats = defaultdict(set)
for a in val_inst['annotations']:
    if a['image_id'] in VAL: val_cats[a['image_id']].add(vcat[a['category_id']])
del val_inst
kb_meta = pd.read_csv(META/'kb_object_metadata.csv')
kb_cats = {int(r.image_id): set(json.loads(r.object_categories)) for r in kb_meta.itertuples()}
pos_to_image = caps['image_id'].astype(int).to_numpy()

val_list = ids['validation']
val_imgs = [load_image(i) for i in val_list]
val_emb = embed_images(val_imgs)

# BLIP caption = the query text the real pipeline will use for BM25
from transformers import BlipProcessor, BlipForConditionalGeneration
bp = BlipProcessor.from_pretrained('Salesforce/blip-image-captioning-base')
bm = BlipForConditionalGeneration.from_pretrained('Salesforce/blip-image-captioning-base').to(device).eval()
blip_q = []
for i in range(0, len(val_imgs), 16):
    inp = bp(images=val_imgs[i:i+16], return_tensors='pt').to(device)
    with torch.inference_mode():
        g = bm.generate(**inp, max_new_tokens=35, num_beams=3)
    blip_q += [bp.decode(x, skip_special_tokens=True).strip() for x in g]
print('example BLIP queries:', blip_q[:3])
assert len(blip_q) == len(val_list)

example BLIP queries: ['a woman cutting up some food on a table', 'a group of people sitting around a dinner table', 'a man in a suit and tie looking at something']


### A4. Run every setup

In [ ]:
# ---------- retrieval primitives ----------
def r_img2img(i):
    _, p = img_index.search(val_emb[i:i+1], 100)
    return [int(img_ids[x]) for x in p[0]]

def r_img2cap(i):
    _, p = txt_index.search(val_emb[i:i+1], 300)
    return dedupe_to_images(p[0], pos_to_image)

def build_bm25(tokenizer):
    t0 = time.perf_counter()
    bm = BM25Okapi([tokenizer(t) or [''] for t in caps['caption_raw'].astype(str)])
    return bm, time.perf_counter() - t0

BM25 = {name: build_bm25(tk) for name, tk in TOKENIZERS.items()}
print({n: round(t, 2) for n, (_, t) in BM25.items()}, 'seconds to index')

cache = {}   # per-image rankings reused across experiments
cache['img2img'] = [r_img2img(i) for i in range(len(val_list))]
cache['img2cap'] = [r_img2cap(i) for i in range(len(val_list))]
for name, (bm, _) in BM25.items():
    cache['bm25:' + name] = [bm25_rank(bm, TOKENIZERS[name], blip_q[i], pos_to_image) for i in range(len(val_list))]

def evaluate(rankings, k=5):
    J, R = [], []
    for i, iid in enumerate(val_list):
        j, r = score_list(rankings[i], val_cats[iid], kb_cats, k)
        J.append(j); R.append(r)
    return np.array(J), np.array(R)

def fused(sources, rrf_k):
    return [rrf([cache[s][i] for s in sources], rrf_k) for i in range(len(val_list))]

CURRENT = 'hybrid_full_rrf60__bm25:current_clean'
configs = {}
configs['image_to_image_only'] = cache['img2img']
configs['image_to_caption_only'] = cache['img2cap']
for n in TOKENIZERS:
    configs[f'bm25_only__{n}'] = cache['bm25:' + n]
configs['dense_only_rrf60'] = fused(['img2img', 'img2cap'], 60)
for n in TOKENIZERS:
    for k in (10, 30, 60, 100):
        configs[f'hybrid_full_rrf{k}__bm25:{n}'] = fused(['img2img', 'img2cap', 'bm25:' + n], k)

{'current_clean': 0.4, 'keep_apostrophes': 0.46, 'no_stopwords': 0.38} seconds to index


### A5. Scores and decision

We switch only if the best setup beats the current one **and** its 95% CI is above zero.

In [ ]:
rows, scores = [], {}
for name, ranked in configs.items():
    J, R = evaluate(ranked)
    scores[name] = J
    rows.append({'config': name, 'jaccard@5': np.nanmean(J), 'cat_recall@5': np.nanmean(R), 'n_images': int((~np.isnan(J)).sum())})
res = pd.DataFrame(rows)

base = scores[CURRENT]
for idx, r in res.iterrows():
    m, lo, hi = paired_bootstrap(scores[r.config] - base)
    res.loc[idx, ['delta_vs_current', 'ci95_low', 'ci95_high']] = [m, lo, hi]
res['significant_vs_current'] = (res.ci95_low > 0) | (res.ci95_high < 0)
res = res.sort_values('jaccard@5', ascending=False).reset_index(drop=True)
res.round(4).to_csv(RELEASE/'validation_selection.csv', index=False)
pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 60)
print(res.round(4).to_string(index=False))

best = res.iloc[0]
decision = {
    'metric': 'object-category Jaccard@5 on 100 validation images (proxy, see notebook text)',
    'current_config': CURRENT,
    'best_config': best.config,
    'best_jaccard@5': float(best['jaccard@5']),
    'current_jaccard@5': float(res.loc[res.config == CURRENT, 'jaccard@5'].iloc[0]),
    'best_beats_current_significantly': bool(best.config != CURRENT and best.significant_vs_current and best.delta_vs_current > 0),
    'recommendation': ('keep current configuration (no significant improvement)'
                       if not (best.config != CURRENT and best.significant_vs_current and best.delta_vs_current > 0)
                       else 'switch to best_config; regenerate BM25 index (KB embeddings unaffected unless the CLIP side changes)'),
    'seed': SEED,
}
(RELEASE/'validation_selection.json').write_text(json.dumps(decision, indent=2))
print(json.dumps(decision, indent=2))

                                   config  jaccard@5  cat_recall@5  n_images  delta_vs_current  ci95_low  ci95_high  significant_vs_current
     hybrid_full_rrf60__bm25:no_stopwords     0.5003        0.8242        98            0.0033   -0.0067     0.0144                   False
     hybrid_full_rrf30__bm25:no_stopwords     0.4995        0.8463        98            0.0025   -0.0086     0.0134                   False
    hybrid_full_rrf100__bm25:no_stopwords     0.4978        0.8264        98            0.0008   -0.0106     0.0133                   False
 hybrid_full_rrf60__bm25:keep_apostrophes     0.4973        0.8382        98            0.0003   -0.0005     0.0013                   False
    hybrid_full_rrf60__bm25:current_clean     0.4970        0.8365        98            0.0000    0.0000     0.0000                   False
 hybrid_full_rrf30__bm25:keep_apostrophes     0.4967        0.8558        98           -0.0002   -0.0079     0.0080                   False
    hybrid_full_rrf3

### Result

The best setup (BM25 without stop words, RRF k = 60) is +0.0033 Jaccard@5, CI [−0.0067, 0.0144].
The CI includes 0, so **we keep the current setup**.

- The full hybrid scores higher than every single source. BM25-only and image→caption-only are significantly worse.
- Adding BM25 to the two dense searches gives +0.007. On 98 scored images this is not significant.
- 2 of the 100 validation images have no COCO objects, so they are left out of the means.

## Part B · Release audit

Only needs Drive (run cell A1 first). It reads files and runs no model.

In [ ]:
# Final release audit. Reads files only: no model, no GPU.
from pathlib import Path
from collections import defaultdict
from itertools import combinations
import json, hashlib, zipfile
import numpy as np
import pandas as pd
import faiss

ROOT = Path('/content/drive/MyDrive/G9_Hybrid_RAG')
DATA = ROOT/'data'
QA = ROOT/'results/qa'
RELEASE = ROOT/'results/preprocessing_release'
RELEASE.mkdir(parents=True, exist_ok=True)

with open(DATA/'splits/split_ids.json') as f: splits = json.load(f)
ids = {s: set(map(int, splits[s])) for s in ['knowledge_base', 'validation', 'test']}
manifest = pd.read_csv(DATA/'splits/image_manifest.csv')
caps = pd.read_csv(DATA/'preprocessed_captions/kb_captions_retrieval.csv')
with open(DATA/'approved_annotations/kb_instances.json') as f: inst = json.load(f)

checks = {}

# --- splits and captions ---
checks['split_counts'] = [len(ids[s]) for s in ids] == [10000, 100, 100]
checks['split_disjoint'] = all(ids[a].isdisjoint(ids[b]) for a, b in combinations(ids, 2))
checks['manifest_ids'] = len(manifest) == 10200 and all(
    set(manifest.loc[manifest.split == s, 'image_id'].astype(int)) == ids[s] for s in ids)
checks['captions_five_each'] = (len(caps) == 50000 and
    set(caps.image_id.astype(int)) == ids['knowledge_base'] and
    caps.groupby('image_id').size().eq(5).all())
checks['kb_annotations_scoped'] = all(a['image_id'] in ids['knowledge_base'] for a in inst['annotations'])

# --- images ---
image_qa = pd.read_csv(QA/'image_preprocessing_audit.csv')
checks['image_preprocessing_pass'] = len(image_qa) == 10200 and image_qa['status'].eq('PASS').all()

# --- embeddings ---
emb = DATA/'embeddings/clip_vit_base_patch32'
image_ids = np.load(emb/'kb_image_ids.npy')
img = np.load(emb/'kb_image_embeddings.npy', mmap_mode='r')
txt = np.load(emb/'kb_text_embeddings.npy', mmap_mode='r')
checks['image_embedding_shape'] = img.shape == (10000, 512) and set(map(int, image_ids)) == ids['knowledge_base']
checks['text_embedding_shape'] = txt.shape == (50000, 512) and (emb/'kb_text_metadata.csv').exists()
checks['embeddings_finite_and_normalized'] = all(
    np.isfinite(block).all() and np.allclose(np.linalg.norm(block, axis=1), 1, atol=2e-3)
    for arr in [img, txt] for pos in range(0, len(arr), 2048)
    for block in [np.asarray(arr[pos:pos+2048])])
align = json.loads((QA/'embedding_alignment_check.json').read_text())
checks['embedding_rows_aligned'] = bool(align.get('PASS'))
cfg = json.loads((emb/'embedding_config.json').read_text())
checks['embedding_config_complete'] = all(k in cfg for k in
    ['model', 'image_batch_size', 'text_batch_size', 'text_max_length', 'transformers_version', 'torch_version'])

# --- indexes ---
idx_dir = DATA/'indexes'
img_ix = faiss.read_index(str(idx_dir/'faiss_images.index'))
txt_ix = faiss.read_index(str(idx_dir/'faiss_captions.index'))
checks['faiss_index_counts'] = img_ix.ntotal == 10000 and txt_ix.ntotal == 50000 and img_ix.d == txt_ix.d == 512
checks['bm25_exists'] = (idx_dir/'bm25.pkl').exists()

# --- archives: names match manifest, no byte-identical image in two splits ---
archive_root = DATA/'images_archives'
name_to_split = {f"{r.split}/{r.file_name}": r.split for r in manifest.itertuples()}
crc_groups, seen_names = defaultdict(list), set()
for split in ids:
    for zp in sorted((archive_root/split).glob('part_*.zip')):
        with zipfile.ZipFile(zp) as z:
            for entry in z.infolist():
                if entry.is_dir(): continue
                assert entry.filename in name_to_split, entry.filename
                assert entry.filename not in seen_names, entry.filename
                seen_names.add(entry.filename)
                crc_groups[(entry.CRC, entry.file_size)].append((split, zp, entry.filename))
checks['archive_manifest_match'] = len(seen_names) == 10200 and seen_names == set(name_to_split)

def file_digest(zip_path, name):
    with zipfile.ZipFile(zip_path) as z:
        return hashlib.sha256(z.read(name)).hexdigest()

cross_split_exact = []
for candidates in crc_groups.values():
    if len({s for s, *_ in candidates}) < 2: continue
    hashes = defaultdict(set)
    for split, zp, name in candidates:
        hashes[file_digest(zp, name)].add(split)
    if any(len(v) > 1 for v in hashes.values()):
        cross_split_exact.append([n for _, _, n in candidates])
checks['no_exact_cross_split_duplicates'] = not cross_split_exact

# --- metadata and validation choice ---
meta_csv = pd.read_csv(DATA/'metadata/image_metadata.csv')
checks['image_metadata_all_10200'] = (len(meta_csv) == 10200 and
    set(meta_csv.image_id.astype(int)) == set().union(*ids.values()))
val_choice = json.loads((RELEASE/'validation_selection.json').read_text())
checks['validation_selection_done'] = 'recommendation' in val_choice

# --- write the release files ---
pending = ['Shared stratified test IDs: size is fixed at 100, but our draw is random, not stratified',
           'Visually verified attribute/context labels, if the teacher requires them',
           'Perceptual near-duplicate detection (only exact byte duplicates are checked)']
summary = {
    'project': 'G9 Hybrid RAG COCO',
    'status': 'READY_FOR_REVIEW_NOT_FROZEN',
    'checks': {k: bool(v) for k, v in checks.items()},
    'all_automated_checks_pass': all(checks.values()),
    'validation_choice': {k: val_choice.get(k) for k in
        ['current_config', 'best_config', 'best_beats_current_significantly', 'recommendation']},
    'known_pending': pending,
    'caption_context_color_source': 'caption keyword heuristics, not image ground truth',
    'spatial_source': 'COCO bbox pair geometry, not semantic relations',
    'exact_cross_split_duplicate_groups': cross_split_exact,
    'files': {
        'split_ids': 'data/splits/split_ids.json',
        'kb_captions': 'data/preprocessed_captions/kb_captions_retrieval.csv',
        'image_emb': 'data/embeddings/clip_vit_base_patch32/kb_image_embeddings.npy',
        'text_emb': 'data/embeddings/clip_vit_base_patch32/kb_text_embeddings.npy',
        'image_index': 'data/indexes/faiss_images.index',
        'text_index': 'data/indexes/faiss_captions.index',
        'bm25': 'data/indexes/bm25.pkl',
        'image_metadata': 'data/metadata/image_metadata.json',
        'validation_selection': 'results/preprocessing_release/validation_selection.json',
    },
}
(RELEASE/'preprocessing_audit.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

small_hashes = {}
for path in [DATA/'splits/split_ids.json', DATA/'splits/image_manifest.csv',
             DATA/'preprocessed_captions/kb_captions_retrieval.csv']:
    small_hashes[str(path.relative_to(ROOT))] = hashlib.sha256(path.read_bytes()).hexdigest()
(RELEASE/'small_file_sha256.json').write_text(json.dumps(small_hashes, indent=2), encoding='utf-8')

readme = [
    '# G9 — Preprocessing release candidate',
    'Not frozen yet: the shared stratified test set is not confirmed. See preprocessing_audit.json.',
    'COCO: 10,000 KB + 100 validation + 100 random test images (seed 42).',
    'KB: 50,000 captions (5 per image). Raw kept for CLIP/evaluation, cleaned copy for BM25.',
    'Vectors: CLIP ViT-B/32, 512-d, L2-normalised. Row alignment verified.',
    'Indexes: FAISS IndexFlatIP (images, captions) + BM25Okapi. Fusion: RRF k=60.',
    f"Validation choice: {val_choice.get('recommendation')}.",
    'Metadata: 10,200 records. Colours/context are caption heuristics; relations are box-overlap proxies.',
    '', '## Automated checks',
    *[f'- {k}: {"PASS" if v else "FAIL"}' for k, v in checks.items()],
    '', '## Pending',
    *[f'- {item}' for item in pending],
    '', 'Do not change split IDs or overwrite embeddings/indexes until all teams agree on the official split.',
]
(RELEASE/'README_preprocessing.md').write_text('\n'.join(readme) + '\n', encoding='utf-8')

print('===== FINAL PREPROCESSING AUDIT =====')
for name, ok in checks.items(): print(('✅' if ok else '❌'), name)
print('\nAutomated checks:', 'PASS' if all(checks.values()) else 'REVIEW FAILURES')
print('Validation choice:', val_choice.get('recommendation'))
print('NOT FROZEN until the official test split is confirmed.')